# Day 20：多模态与嵌入模型：视觉 API 封装、Embedding 向量与余弦相似度（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 3 周**：第3周：大模型底层原理、Prompt 工程与工具调用实战  
> **GitLab 工单**：`STARMIND-020` ｜ **开发分支**：`feat/day20-multimodal-embedding`  
> **配套 HTML5 课件**：`courseware/day20/index.html` ｜ **独立版本快照**：`versions/day20/`

---

## 🎙️ 老陈开场旁白

早啊各位！昨天咱们给星辰智脑装上了手，今天给它装上「眼睛」和「语义大脑」！先说眼睛——**多模态大模型（Vision LLM & 文生图）**，在 OpenAI / Qwen-VL / GLM-4V 协议里，传图片本质上是把图片转成 Base64 Data URI 塞进 `content` 数组里的 `image_url` 字段；再说语义大脑——**Embedding（文本嵌入向量）**，它能把一句话压缩成一串高维浮点数向量，语义越接近的句子，向量夹角越小、**余弦相似度越接近 1.0**！这就是后面第三阶段企业级 RAG 知识库的绝对地基。今天咱们亲手把多模态客户端和纯 Python 余弦相似度语义匹配器全部跑通！

### 承上启下
- **昨日基线**：昨天（Day 19）我们通过 Function Calling 给大模型装上了双手，成功打通了天气、计算器与内部知识库调用闭环。但客服同学反馈了两个新难题：第一，很多客户遇到报错懒得打字，直接甩一张「报错截图」过来；第二，昨天咱们的知识库检索只能靠精确关键词匹配，员工问「出差住酒店能报多少钱」时，因为没出现「差旅」二字居然搜不到「差旅住宿报销标准」！
- **明日预告**：明天（Day 21）是第 3 周的周测 + 综合练习日，我们将把本周学的 Prompt、防注入、多轮对话、Function Calling 与流式输出全部拧成一股绳，打造综合引擎并通过自动化周测！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day20'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-020）

星辰智脑客服与 IT 支持每天收到大量报错截图与口语化提问（如「住酒店报销上限」「电脑连不上公司内网」），纯文本与字面关键词检索召回率不足 60%。

**今日迭代目标：**
- 封装标准多模态客户端 `MultimodalClient`，支持图片 Base64 Data URI 编码、Vision 图像理解请求构造与文生图（Image Generation）参数封装。
- 实现纯 Python 语义向量化引擎（融合字符 n-gram 与企业同义词语义空间投影）及 L2 归一化、余弦相似度计算。
- 构建 FAQ 相似问题语义匹配器，支持 Top-K 召回与最低相似度阈值拒答（Fallback）。

**验收标准：**
- 多模态客户端严格校验 PNG/JPEG/WEBP 魔数或格式并拒绝非法超大负载，构造的 `messages` 100% 符合 OpenAI Vision 协议。
- 相同向量余弦相似度为 `1.0`，正交向量为 `0.0`，反向向量为 `-1.0`；任意非零向量 L2 归一化后模长严格为 `1.0`。
- 语义匹配器能准确将「出差住酒店最多报多少」匹配至「差旅住宿报销标准」，并将完全无关的闲聊（如「今晚吃火锅」）按阈值安全拒答。

---

## 📚 课堂核心技术笔记

### 1. 1. 多模态 Vision API 的 detail 参数与图片 Token 计费

调用视觉理解模型（如 Qwen-VL-Max、GPT-4o）时，`image_url` 支持 `detail: 'low'`（低清快速，固定约 85 Token，适合图标分类）和 `detail: 'high'`（高清切片，按 512x512 网格切块计费，适合 OCR 识别报错截图和发票文字）。发送前将超大图片等比缩放到长边 <= 1024px，既能看清文字又能省下 60% 视觉 Token。

> **⚠️ 老陈避坑**：常见坑：直接传内网 URL（如 `http://192.168.1.10/err.png`）给公网大模型 API，公网服务器根本访问不到你公司内网，必定报图片下载超时！内网图片必须转成 Base64 Data URI 发送。

### 2. 2. 为什么语义检索首选余弦相似度而不是欧氏距离？

余弦相似度衡量的是两个向量在方向上的一致性，不受文本长度导致的向量模长波动影响。并且一旦向量全部完成 L2 归一化，余弦相似度与欧氏距离的排序完全等价（$\|A-B\|^2 = 2 - 2(A \cdot B)$），向量数据库（如 Chroma / Milvus / Qdrant）底层都依赖这一数学性质加速检索。

> **⚠️ 老陈避坑**：绝对禁忌：建索引时用了 A 嵌入模型（如 `text-embedding-v3`，1024 维），查询时却换成了 B 嵌入模型（如 `bge-large-zh`，1024 维）——即使维度相同，不同模型的向量空间完全不对齐，检索结果将彻底乱套！

### 3. 3. 国产主流 Embedding 模型选型参考

国内企业落地首选：开源私有化推荐 **BGE-M3**（支持中英多语言、8192 长文本、稠密+稀疏混合检索）或 **BCEmbedding**；云端 API 推荐阿里云百炼 `text-embedding-v3` 或智谱 `embedding-3`，性价比极高。

> **⚠️ 老陈避坑**：FAQ 匹配时，把「标准问题」拿去做 Embedding 建索引通常比把「长篇答案」建索引命中率更高，因为用户提问的句式更接近标准问（Question-to-Question Matching）。


## 💻 实操步骤 1：步骤一：封装多模态图像理解与文生图 API 客户端（MultimodalClient）

- **目标文件**：`versions/day20/src/day20_multimodal_client.py`
- **老陈代码旁白**：咱们先写 `src/day20_multimodal_client.py`，把图片安全校验、Base64 Data URI 转换、OpenAI 兼容 Vision 图像理解请求构造以及文生图参数封装做成一套企业级客户端，离线也能完整跑通测试。


In [ ]:
"""
StarMind Day 20 - 步骤一：多模态图像理解（Vision）与文生图（Image Generation）API 客户端封装
支持 MIME 白名单、体积限制、Base64 Data URI 构造与离线确定性推理桩。
"""

import base64
from dataclasses import dataclass
from typing import Any, Dict, List


ALLOWED_MIME_TYPES = {"image/png", "image/jpeg", "image/webp"}
ALLOWED_IMAGE_SIZES = {"512x512", "1024x1024", "1024x1792", "1792x1024"}


@dataclass
class VisionResult:
    """图像理解分析结果。"""
    model: str
    summary: str
    detected_tags: List[str]
    request_payload: Dict[str, Any]


@dataclass
class ImageGenResult:
    """文生图结果。"""
    model: str
    prompt: str
    size: str
    revised_prompt: str
    mock_image_url: str


class MultimodalClient:
    """星辰智脑多模态 API 统一客户端。"""

    def __init__(self, max_image_bytes: int = 5 * 1024 * 1024) -> None:
        self.max_image_bytes = max_image_bytes

    def encode_image_to_data_uri(self, image_bytes: bytes, mime_type: str = "image/png") -> str:
        """校验图片格式与大小上限，编码为标准 Base64 Data URI。"""
        if mime_type not in ALLOWED_MIME_TYPES:
            raise ValueError(f"不支持的图片 MIME 类型: {mime_type}，仅允许 {ALLOWED_MIME_TYPES}")
        if not image_bytes:
            raise ValueError("图片字节内容不能为空")
        if len(image_bytes) > self.max_image_bytes:
            raise ValueError(f"图片体积 {len(image_bytes)} 字节超出上限 {self.max_image_bytes} 字节")

        b64 = base64.b64encode(image_bytes).decode("ascii")
        return f"data:{mime_type};base64,{b64}"

    def build_vision_payload(
        self,
        prompt: str,
        image_bytes: bytes,
        mime_type: str = "image/png",
        detail: str = "high",
        model: str = "qwen-vl-max",
    ) -> Dict[str, Any]:
        """构造 100% 符合 OpenAI Vision 协议标准的请求 JSON 体。"""
        if detail not in {"low", "high", "auto"}:
            raise ValueError(f"非法的 detail 参数: {detail}")
        data_uri = self.encode_image_to_data_uri(image_bytes, mime_type=mime_type)
        return {
            "model": model,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": data_uri, "detail": detail}},
                    ],
                }
            ],
        }

    def analyze_image(
        self,
        prompt: str,
        image_bytes: bytes,
        mime_type: str = "image/png",
        detail: str = "high",
        model: str = "qwen-vl-max",
    ) -> VisionResult:
        """执行多模态图像理解（离线仿真分析，返回标准结构与完整请求体）。"""
        payload = self.build_vision_payload(prompt, image_bytes, mime_type, detail, model)
        tags = ["enterprise_screenshot"]
        if "报错" in prompt or "错误" in prompt or "502" in prompt:
            tags.extend(["error_dialog", "sre_diagnostic"])
            summary = "已识别界面报错截图：检测到网关 502 Bad Gateway 提示，建议优先检查上游服务健康检查与超时配置。"
        elif "发票" in prompt:
            tags.extend(["invoice_ocr", "finance"])
            summary = "已提取电子发票关键字段：抬头为杭州星辰数科有限公司，税率 6%，状态核验正常。"
        else:
            tags.append("general_vision")
            summary = f"已完成图像理解（清晰度={detail}，大小={len(image_bytes)}B）：画面主体清晰，符合业务提交规范。"

        return VisionResult(model=model, summary=summary, detected_tags=tags, request_payload=payload)

    def generate_image(
        self,
        prompt: str,
        size: str = "1024x1024",
        style: str = "corporate_flat",
        model: str = "wanx-v2",
    ) -> ImageGenResult:
        """封装文生图 API 调用与企业提示词增强。"""
        if not prompt.strip():
            raise ValueError("文生图 prompt 不能为空")
        if size not in ALLOWED_IMAGE_SIZES:
            raise ValueError(f"不支持的图片分辨率: {size}，可选 {ALLOWED_IMAGE_SIZES}")

        revised = f"{prompt.strip()}，风格：{style}，高清无水印，企业级科技蓝配色"
        slug = abs(hash(revised)) % 100000
        return ImageGenResult(
            model=model,
            prompt=prompt,
            size=size,
            revised_prompt=revised,
            mock_image_url=f"https://assets.starmind.local/generated/{model}_{size}_{slug}.png",
        )


if __name__ == "__main__":
    client = MultimodalClient()
    fake_png = b"\x89PNG\r\n\x1a\n" + b"starmind-demo-image-payload"
    v_res = client.analyze_image("请帮我分析这张 502 报错截图的原因", fake_png, mime_type="image/png")
    print("视觉理解结果:", v_res.summary, "| 标签:", v_res.detected_tags)
    g_res = client.generate_image("星辰智脑 AI 平台发布会主视觉海报", size="1024x1024")
    print("文生图结果:", g_res.mock_image_url)


### 🔍 步骤 1 代码深度解析

1. `encode_image_to_data_uri` 把 MIME 白名单和体积上限校验做在编码之前，防止非法文件或超大文件拖垮网关内存。
2. `build_vision_payload` 生成的字典结构完全对齐 OpenAI / 通义千问 VL / GLM-4V 的多模态标准，上线时直接 `httpx.post(url, json=payload)` 即可。
3. `generate_image` 校验分辨率枚举并自动补全企业风格增强词（`revised_prompt`）。

**预期运行输出参考：**
```text
视觉理解结果: 已识别界面报错截图：检测到网关 502 Bad Gateway 提示，建议优先检查上游服务健康检查与超时配置。 | 标签: ['enterprise_screenshot', 'error_dialog', 'sre_diagnostic']
文生图结果: https://assets.starmind.local/generated/wanx-v2_1024x1024_...
```


## 💻 实操步骤 2：步骤二：实现纯 Python Embedding 向量化、余弦相似度与 FAQ 语义匹配工具

- **目标文件**：`versions/day20/src/day20_semantic_matcher.py`
- **老陈代码旁白**：接下来写 `src/day20_semantic_matcher.py`！我们纯手工实现 L2 归一化、余弦相似度公式，并打造一个融合「企业同义词概念槽 + 字符双字 n-gram 哈希投影」的轻量级离线 Embedding 模型，让「出差住酒店最多报多少」精准命中「差旅住宿报销标准」！


In [ ]:
"""
StarMind Day 20 - 步骤二：Embedding 文本向量化、余弦相似度计算与 FAQ 相似问题语义匹配工具
纯 Python 零外部依赖实现，完美呈现语义向量检索与阈值拒答机制。
"""

import hashlib
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple


def l2_normalize(vector: List[float]) -> List[float]:
    """对向量执行 L2 归一化，使其欧氏模长严格等于 1.0。"""
    norm = math.sqrt(sum(x * x for x in vector))
    if norm == 0.0:
        return [0.0 for _ in vector]
    return [x / norm for x in vector]


def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
    """计算两个等长向量的余弦相似度，返回值域 [-1.0, 1.0]。"""
    if len(vec_a) != len(vec_b):
        raise ValueError(f"向量维度不一致: {len(vec_a)} vs {len(vec_b)}")
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return max(-1.0, min(1.0, dot / (norm_a * norm_b)))


class StarMindLocalEmbedder:
    """
    轻量级确定性语义嵌入模型（离线仿真 BGE / text-embedding-v3 行为）：
    - 前 16 维：企业领域语义概念投影（打通同义词鸿沟，如 住酒店 <-> 差旅住宿）
    - 后 48 维：字符 unigram + bigram 哈希投影（保留细粒度字面特征）
    总维度 = 64 维，输出自动 L2 归一化。
    """

    CONCEPT_GROUPS: List[Tuple[str, ...]] = [
        ("差旅", "出差", "酒店", "住宿", "报销", "旅费", "发票"),
        ("vpn", "内网", "远程", "连不上", "办公网", "网络权限"),
        ("请假", "年假", "病假", "调休", "考勤", "休假"),
        ("发票", "专票", "税号", "抬头", "开票", "财务"),
        ("密码", "重置", "登录", "账号", "锁定", "sso"),
        ("薪资", "工资", "发薪", "社保", "公积金", "个税"),
        ("合同", "法务", "盖章", "用印", "审批", "协议"),
        ("报障", "报错", "502", "超时", "故障", "崩溃", "bug"),
    ]

    def __init__(self, hash_dims: int = 48) -> None:
        self.concept_dims = len(self.CONCEPT_GROUPS)
        self.hash_dims = hash_dims
        self.dim = self.concept_dims + self.hash_dims

    def embed(self, text: str) -> List[float]:
        """将任意中英文文本编码为 L2 归一化的 64 维稠密语义向量。"""
        clean = text.strip().lower()
        vec = [0.0] * self.dim
        if not clean:
            return vec

        # 1. 语义概念槽加权投影（高权重，实现同义词语义对齐）
        for idx, group in enumerate(self.CONCEPT_GROUPS):
            hits = sum(1 for kw in group if kw in clean)
            if hits > 0:
                vec[idx] += 3.0 * hits

        # 2. 字符级 bigram 哈希投影（保留局部字面匹配能力）
        compact = "".join(clean.split())
        grams = [compact[i : i + 2] for i in range(max(1, len(compact) - 1))]
        for g in grams:
            digest = hashlib.md5(g.encode("utf-8")).digest()
            slot = self.concept_dims + (digest[0] % self.hash_dims)
            vec[slot] += 0.6

        return l2_normalize(vec)


@dataclass
class FAQEntry:
    faq_id: str
    question: str
    answer: str
    category: str


@dataclass
class MatchHit:
    faq_id: str
    question: str
    answer: str
    score: float


class SemanticFAQMatcher:
    """基于 Embedding + 余弦相似度的企业 FAQ 相似问题语义匹配器。"""

    def __init__(self, embedder: StarMindLocalEmbedder | None = None) -> None:
        self.embedder = embedder or StarMindLocalEmbedder()
        self._entries: List[FAQEntry] = []
        self._vectors: List[List[float]] = []

    def add_faqs(self, entries: List[FAQEntry]) -> None:
        """批量建立 FAQ 向量索引。"""
        for entry in entries:
            self._entries.append(entry)
            self._vectors.append(self.embedder.embed(entry.question))

    def search(self, query: str, top_k: int = 3, min_score: float = 0.35) -> List[MatchHit]:
        """执行 Top-K 语义检索并按 min_score 阈值过滤低相关噪声。"""
        q_vec = self.embedder.embed(query)
        scored: List[Tuple[float, FAQEntry]] = []
        for entry, doc_vec in zip(self._entries, self._vectors):
            sim = cosine_similarity(q_vec, doc_vec)
            if sim >= min_score:
                scored.append((sim, entry))

        scored.sort(key=lambda item: item[0], reverse=True)
        return [
            MatchHit(
                faq_id=e.faq_id,
                question=e.question,
                answer=e.answer,
                score=round(s, 4),
            )
            for s, e in scored[:top_k]
        ]


if __name__ == "__main__":
    matcher = SemanticFAQMatcher()
    matcher.add_faqs([
        FAQEntry("FAQ-01", "杭州与一线城市差旅住宿报销标准是多少？", "北京/上海/深圳/杭州住宿上限 600 元/晚。", "finance"),
        FAQEntry("FAQ-02", "新员工如何申请公司内网 VPN 远程访问？", "请在 OA 发起 IT 权限申请，主管审批后 30 分钟开通。", "it"),
        FAQEntry("FAQ-03", "年假和病假怎么在考勤系统里提交请假？", "登录星辰 OA-考勤休假模块选择对应假别提交即可。", "hr"),
        FAQEntry("FAQ-04", "SSO 单点登录密码忘记了如何重置？", "访问门户登录页点击忘记密码并通过企业微信扫码验证。", "it"),
    ])

    for q in ["出差住酒店最多能报多少钱？", "在家连不上公司办公网怎么办？", "今晚杭州西湖附近吃什么火锅好？"]:
        hits = matcher.search(q, top_k=2, min_score=0.35)
        print(f"提问: {q}")
        if hits:
            for h in hits:
                print(f"  -> [{h.score:.4f}] {h.question} | 答: {h.answer}")
        else:
            print("  -> [低于阈值拒答] 未命中企业 FAQ，建议转人工或通用对话。")


### 🔍 步骤 2 代码深度解析

1. `l2_normalize` 与 `cosine_similarity` 完全手写数学公式，并做了零向量防护与 `[-1.0, 1.0]` 浮点误差截断。
2. `StarMindLocalEmbedder` 巧妙结合领域语义概念槽与字符 bigram 哈希投影，无需下载几百 MB 的 PyTorch 权重就能在本地真实展现「同义词高相似、无关句低相似」的语义空间特性。
3. `SemanticFAQMatcher.search` 通过 `min_score` 阈值把无关闲聊拒之门外，避免硬凑答案。

**预期运行输出参考：**
```text
提问: 出差住酒店最多能报多少钱？
  -> [0.95xx] 杭州与一线城市差旅住宿报销标准是多少？ | 答: 北京/上海/深圳/杭州住宿上限 600 元/晚。
提问: 今晚杭州西湖附近吃什么火锅好？
  -> [低于阈值拒答] 未命中企业 FAQ，建议转人工或通用对话。
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day20/tests/test_day20.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 20 自动化单元测试：验证多模态客户端、余弦相似度数学精度与 FAQ 语义匹配。"""

import math
import pytest
from src.day20_multimodal_client import MultimodalClient
from src.day20_semantic_matcher import (
    FAQEntry,
    SemanticFAQMatcher,
    StarMindLocalEmbedder,
    cosine_similarity,
    l2_normalize,
)


def test_multimodal_client_vision_and_validation():
    client = MultimodalClient(max_image_bytes=1024)
    png_bytes = b"\x89PNG\r\n\x1a\nfake-image"
    res = client.analyze_image("请分析这个 502 报错截图", png_bytes, mime_type="image/png", detail="high")
    assert "502" in res.summary
    content_blocks = res.request_payload["messages"][0]["content"]
    assert content_blocks[1]["type"] == "image_url"
    assert content_blocks[1]["image_url"]["url"].startswith("data:image/png;base64,")

    # 非法 MIME 与超体积应抛 ValueError
    with pytest.raises(ValueError):
        client.encode_image_to_data_uri(png_bytes, mime_type="application/exe")
    with pytest.raises(ValueError):
        client.encode_image_to_data_uri(b"x" * 2048, mime_type="image/png")

    gen = client.generate_image("星辰智脑科技蓝架构海报", size="1024x1024")
    assert gen.size == "1024x1024"
    assert gen.mock_image_url.startswith("https://")


def test_cosine_similarity_and_l2_normalization():
    v = l2_normalize([3.0, 4.0])
    assert math.isclose(math.sqrt(sum(x * x for x in v)), 1.0, rel_tol=1e-6)
    assert cosine_similarity([1.0, 0.0], [1.0, 0.0]) == pytest.approx(1.0)
    assert cosine_similarity([1.0, 0.0], [0.0, 1.0]) == pytest.approx(0.0)
    assert cosine_similarity([1.0, 0.0], [-1.0, 0.0]) == pytest.approx(-1.0)
    with pytest.raises(ValueError):
        cosine_similarity([1.0, 2.0], [1.0])


def test_semantic_faq_matcher_synonym_hit_and_rejection():
    matcher = SemanticFAQMatcher(StarMindLocalEmbedder())
    matcher.add_faqs([
        FAQEntry("FAQ-01", "差旅住宿报销标准是多少？", "一线城市 600 元/晚。", "finance"),
        FAQEntry("FAQ-02", "如何申请公司内网 VPN 权限？", "OA 提交申请。", "it"),
        FAQEntry("FAQ-03", "考勤系统如何请年假？", "OA 考勤模块提交。", "hr"),
    ])

    # 同义词口语问法（不含"差旅"和"标准"）应精准命中 FAQ-01
    hits = matcher.search("出差住酒店能报多少钱", top_k=1, min_score=0.35)
    assert len(hits) == 1
    assert hits[0].faq_id == "FAQ-01"
    assert hits[0].score > 0.7

    # 完全无关的闲聊应被阈值过滤拒答
    irrelevant = matcher.search("今天西湖边的桂花开了吗", top_k=3, min_score=0.5)
    assert irrelevant == []

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：批量文本去重器（Semantic Deduplicator）

**题目要求：**
客服知识库常被不同运营重复录入意思几乎一样的问题。请使用 `StarMindLocalEmbedder` 和 `cosine_similarity` 编写 `deduplicate_questions(questions, threshold=0.85)`，过滤掉与已保留问题相似度 >= `threshold` 的重复问句。

**思路提示：** 遍历问题列表，逐一计算其向量与已保留向量列表的最大余弦相似度，若均低于 `threshold` 则保留。


In [ ]:
from typing import List
from src.day20_semantic_matcher import StarMindLocalEmbedder, cosine_similarity


def deduplicate_questions(questions: List[str], threshold: float = 0.85) -> List[str]:
    """基于余弦相似度的语义去重器。"""
    embedder = StarMindLocalEmbedder()
    kept_texts: List[str] = []
    kept_vecs: List[List[float]] = []

    for q in questions:
        vec = embedder.embed(q)
        if all(cosine_similarity(vec, kv) < threshold for kv in kept_vecs):
            kept_texts.append(q)
            kept_vecs.append(vec)
    return kept_texts


if __name__ == "__main__":
    raw = [
        "差旅住宿报销标准是多少？",
        "出差住酒店报销上限多少？",
        "新员工怎么申请内网VPN？",
    ]
    print("去重结果:", deduplicate_questions(raw, threshold=0.80))


**答案解析与评分标准：**

语义去重是构建高质量企业 RAG 知识库前最重要的数据清洗步骤之一，能避免检索 Top-3 全是重复段落。


### 作业 2（企业实战）：向量检索评估指标 MRR（Mean Reciprocal Rank）计算器

**题目要求：**
给定一组测试查询 `(query, expected_faq_id)`，使用 `SemanticFAQMatcher` 检索 Top-3，若标准答案排在第 `rank` 位（1-indexed）则该条得分为 `1/rank`，未命中得 `0`，编写函数计算整个评测集的平均倒数排名 MRR。

**思路提示：** 遍历检索结果 `enumerate(hits, 1)`，找到 `hit.faq_id == expected_id` 时累加 `1.0 / rank`。


In [ ]:
from typing import List, Tuple
from src.day20_semantic_matcher import FAQEntry, SemanticFAQMatcher


def evaluate_mrr(matcher: SemanticFAQMatcher, test_cases: List[Tuple[str, str]], top_k: int = 3) -> float:
    """计算语义检索系统的 MRR (Mean Reciprocal Rank) 指标。"""
    if not test_cases:
        return 0.0
    total_rr = 0.0
    for query, expected_id in test_cases:
        hits = matcher.search(query, top_k=top_k, min_score=0.0)
        for rank, hit in enumerate(hits, 1):
            if hit.faq_id == expected_id:
                total_rr += 1.0 / rank
                break
    return round(total_rr / len(test_cases), 4)


if __name__ == "__main__":
    m = SemanticFAQMatcher()
    m.add_faqs([
        FAQEntry("F1", "差旅住宿报销标准", "600元/晚", "fin"),
        FAQEntry("F2", "内网VPN申请流程", "OA提交", "it"),
    ])
    print("MRR:", evaluate_mrr(m, [("出差住酒店报多少", "F1"), ("连不上公司内网", "F2")]))


**答案解析与评分标准：**

MRR 是搜索与 RAG 领域最常用的排序质量指标，MRR 越接近 1.0 说明正确答案越稳定排在第一位。


### 作业 3（架构进阶）：混合检索分数融合器（RRF 倒数排名融合）

**题目要求：**
在企业检索中常把「关键词 BM25 排名列表」和「向量语义排名列表」融合。请实现经典的 Reciprocal Rank Fusion：`rrf_fuse(list_a, list_b, k=60)`，每篇文档得分公式为 $\sum \frac{1}{k + \text{rank}}$，按总分降序返回文档 ID 列表。

**思路提示：** 遍历两个排序列表（rank 从 1 开始），用字典累加每个 `doc_id` 的 `1.0 / (k + rank)`。


In [ ]:
from typing import Dict, List, Tuple


def rrf_fuse(list_a: List[str], list_b: List[str], k: int = 60) -> List[Tuple[str, float]]:
    """Reciprocal Rank Fusion (RRF) 混合检索多路排名融合算法。"""
    scores: Dict[str, float] = {}
    for ranked_list in (list_a, list_b):
        for rank, doc_id in enumerate(ranked_list, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(((d, round(s, 6)) for d, s in scores.items()), key=lambda x: x[1], reverse=True)


if __name__ == "__main__":
    bm25_top = ["DOC-2", "DOC-1", "DOC-5"]
    vector_top = ["DOC-1", "DOC-3", "DOC-2"]
    print("RRF 融合排名:", rrf_fuse(bm25_top, vector_top))


**答案解析与评分标准：**

RRF 不依赖两路检索分数的量纲归一化，仅凭排名就能稳定融合关键词精确匹配与向量语义召回，是企业级混合检索的黄金标准。


---

## 🎯 老陈收尾复盘

今天这一课直接为第三阶段的 RAG 埋下了最关键的伏笔！记住两个核心点：第一，多模态传图务必校验 MIME 类型和文件大小上限（防止一张 20MB 原图把接口内存打爆）；第二，Embedding 检索的核心是「向量归一化 + 余弦相似度 + 置信度阈值拒答」，搜不到相关文档时宁可拒答也别硬凑胡编！
